# Dependencies and Imports


In [ ]:
# !pip3 install torch pytorch-lightning multiprocess ray tensorboardx

In [13]:
import os
os.environ["RAY_TRAIN_V2_ENABLED"] = "1"
os.environ["TUNE_WARN_EXCESSIVE_EXPERIMENT_CHECKPOINT_SYNC_THRESHOLD_S"] = "0" # Disable excessive checkpoint sync warning
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F # ReLU, Softmax
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader, TensorDataset #, IterableDataset, TensorDataset
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

# from lightning.pytorch.tuner import Tuner
import lightning.pytorch as pl
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint, EarlyStopping
# from lightning.pytorch.loggers import TensorBoardLogger, WandbLogger
import ray
from ray.tune.integration.pytorch_lightning import TuneReportCallback, TuneReportCheckpointCallback
from functools import partial
import numpy as np
import pandas as pd
import datetime as dt
from dateutil.relativedelta import relativedelta
# from datetime import date, timedelta

# from NowcastingPipelineM import NowcastingPH_M
# import dynamicfactoranalysis.dynamicfactoranalysis as dfa
from STSeq2One import STMFSeq2One, STMFSeq2OneLightning
from MTSeq2One import MTMFSeq2One, MTMFSeq2OneLightning #MTMFSeq2OneOLD, MTMFSeq2OneLightningOLD
# from Seq2OneM import MTSeq2OneM, MTSeq2OneMLightning
from data_utils import NowcastingLSTM_MQ, sliding_windows_ST, sliding_windows_MT # get_dataloader_for_vintage, 
from train_vintage import train_model
from evaluate_vintage import evaluate_one_vintage_ST, evaluate_one_vintage_MT

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device="cpu"
# print(device)
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.current_device())
print(torch.cuda.device(0))
print(torch.cuda.get_device_name(0))

2.6.0+cu124
True
1
0
Quadro RTX 6000


# Utils

In [ ]:
# def set_seed(seed):
#     import random
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)
#     # torch.backends.cudnn.deterministic = True                     # Only cuDNN convolution algorithms
#     torch.use_deterministic_algorithms(True)                        # All torch and cuDNN algorithms when available. RunTime error if not available.
#     torch.backends.cudnn.benchmark = False                          # Uses the same algorthim. May lose out on performance.
#     # torch.utils.deterministic.fill_uninitialized_memory = True    # For torch.empty() or torch.Tensor.resize()
#     os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'                 # For RNN/LSTM


In [5]:
model = STMFSeq2One(dim_x=11,dim_y=1, num_layers=1)
print("Parameters: ", sum(p.numel() for p in model.parameters() if p.requires_grad))
model = MTMFSeq2One(dim_x=11, dim_y=1, num_layers=1)
print("Parameters: ", sum(p.numel() for p in model.parameters() if p.requires_grad))
# model = MTSeq2OneM(dim_x=11, dim_y=1, num_layers=1)
# print("Parameters: ", sum(p.numel() for p in model.parameters() if p.requires_grad))
model = MTMFSeq2OneOLD(dim_x=11, dim_y=1, num_layers=1)
print("Parameters: ", sum(p.numel() for p in model.parameters() if p.requires_grad))

Parameters:  1458
Parameters:  1533
Parameters:  1533


In [2]:
%reload_ext tensorboard
%tensorboard --logdir="lightning_logs" --port=6006

Launching TensorBoard...

In [49]:
import os
import shutil

# Define the directories
tb_logs_dir = "tb_logs"
checkpoint_dir = "checkpoints"

# Delete version 1 logs in tb_logs
if os.path.exists(tb_logs_dir):
    for item in os.listdir(tb_logs_dir):
        if "version_1" in os.listdir(tb_logs_dir + '/' + item):
            path = os.path.join(tb_logs_dir, item, "version_1")
            if os.path.isdir(path):
                shutil.rmtree(path)
                print(f"Deleted: {path}")

# Delete model-v1 directories in checkpoint
if os.path.exists(checkpoint_dir):
    for item in os.listdir(checkpoint_dir):
        if "model-v1.ckpt" in os.listdir(checkpoint_dir + '/' + item):
            path = os.path.join(checkpoint_dir, item, "model-v1.ckpt")
            os.remove(path)
            print(f"Deleted: {path}")

Deleted: tb_logs/vintage_2017-11-30 00:00:00/version_1
Deleted: tb_logs/vintage_2021-10-31 00:00:00/version_1
Deleted: tb_logs/vintage_2018-12-31 00:00:00/version_1
Deleted: tb_logs/vintage_2019-04-30 00:00:00/version_1
Deleted: tb_logs/vintage_2021-09-30 00:00:00/version_1
Deleted: tb_logs/vintage_2022-03-31 00:00:00/version_1
Deleted: tb_logs/vintage_2021-08-31 00:00:00/version_1
Deleted: tb_logs/vintage_2020-03-31 00:00:00/version_1
Deleted: tb_logs/vintage_2022-08-31 00:00:00/version_1
Deleted: tb_logs/vintage_2018-04-30 00:00:00/version_1
Deleted: tb_logs/vintage_2021-01-31 00:00:00/version_1
Deleted: tb_logs/vintage_2020-12-31 00:00:00/version_1
Deleted: tb_logs/vintage_2022-07-31 00:00:00/version_1
Deleted: tb_logs/vintage_2019-01-31 00:00:00/version_1
Deleted: tb_logs/vintage_2018-08-31 00:00:00/version_1
Deleted: tb_logs/vintage_2018-11-30 00:00:00/version_1
Deleted: tb_logs/vintage_2018-05-31 00:00:00/version_1
Deleted: tb_logs/vintage_2017-10-31 00:00:00/version_1
Deleted: t

In [ ]:
import os
import shutil

ray_results_dir = "/home/btiu/ray_results"

if os.path.exists(ray_results_dir):
    for item in os.listdir(ray_results_dir):
        item_path = os.path.join(ray_results_dir, item)
        if os.path.isdir(item_path) and item.startswith("train"):
            shutil.rmtree(item_path)
            print(f"Deleted: {item_path}")
else:
    print(f"Directory does not exist: {ray_results_dir}")

# Single output LSTM

In [45]:
target = 'PHL_GDP_SA'
vintage = pd.to_datetime('2020-06-30')
# kmpair = {'PE': ['CRVADER_BVN','CR_BxP_0'],'PU+': ['CRVADER_BVN','CR_BxP_0']} # kmpair = {'PE':['CR_B0']}
kmpair = {'PE': ['VADERstanceweight_log_stl'], 'PU+': ['CR_lognorm']}
data_window = 6
with_econ = True
with_tweets = True

In [46]:
### Train & Val Data
data_model = NowcastingLSTM_MQ()
data, target_scaler, econ_scaler, tweets_scaler = data_model.load_data(vintage=vintage,window=1000, kmpair=kmpair, with_econ=with_econ, with_tweets=with_tweets, target_release_lag=True,scaled=True, extend=False, DFM_order=(1,0,1,0), optimize_order = False, target=target)
data = data.dropna()
train_data = data.iloc[:-3]
val_data = data.iloc[-data_window-3:]
# val_data, _, _, _ = data_model.load_data(vintage=vintage+pd.offsets.QuarterEnd(0)+pd.DateOffset(months=2),window=1000, kmpair=kmpair, with_econ=with_econ, with_tweets=with_tweets, target_release_lag=False, scaled=False, extend=False, DFM_order=(1,0,1,0), optimize_order = False, target=target)
# val_data = val_data.dropna().iloc[-data_window-3:]
# display(val_data)
# val_data.iloc[:,:1] = target_scaler.transform(val_data.iloc[:,:1])
# if with_econ and with_tweets:
#     econ_n_feat = econ_scaler.n_features_in_
#     tweets_n_feat = tweets_scaler.n_features_in_
#     val_data.iloc[:,1:tweets_n_feat+1] = tweets_scaler.transform(val_data.iloc[:,1:tweets_n_feat+1])
#     val_data.iloc[:,tweets_n_feat+1:] = econ_scaler.transform(val_data.iloc[:,tweets_n_feat+1:])
# elif with_econ:
#     econ_n_feat = econ_scaler.n_features_in_
#     val_data.iloc[:,1:econ_n_feat+1] = econ_scaler.transform(val_data.iloc[:,1:econ_n_feat+1])
# elif with_tweets:
#     tweets_n_feat = tweets_scaler.n_features_in_
#     val_data.iloc[:,1:tweets_n_feat+1] = tweets_scaler.transform(val_data.iloc[:,1:tweets_n_feat+1])

display(train_data)
display(val_data)

trainX_in, trainY_in, _, trainY_out = sliding_windows_ST(train_data, train=True, seq_length=data_window)
valX_in, valY_in, _, valY_out = sliding_windows_ST(val_data, train=True, seq_length=data_window)
print("Train X shape: ", trainX_in.shape, "Train Y shape: ", trainY_in.shape, "Train Y out shape: ", trainY_out.shape)
print("Val X shape: ", valX_in.shape, "Val Y shape: ", valY_in.shape, "Val Y out shape: ", valY_out.shape)
train_dataset = TensorDataset(trainX_in, trainY_in, trainY_out)
val_dataset = TensorDataset(valX_in, valY_in, valY_out)
train_loader = DataLoader(train_dataset, batch_size=200, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=200, shuffle=False)

# get_dataloader_for_vintage(pd.to_datetime('2017-01-31'), with_econ=True, with_tweets=False, kmpair = kmpair, data_window = 15, task='singletask')

,target,TWT.VADERstanceweight_log_stl_PE,TWT.CR_lognorm_PU+,ECN.PPI_MoMlog,ECN.CPI_MoMlog,ECN.IPI_MoMlog,ECN.exports_MoMlog,ECN.imports_MoMlog,ECN.govtexpt_MoMlog,ECN.pseiclose_MoMlog,ECN.phpusd_MoMlog,ECN.m1_MoMlog,ECN.tbill_usd90_diff,ECN.tdr_php360_diff
date,,,,,,,,,,,,,,
2010-01,1.473980,0.260375,-0.766140,-0.194447,0.065249,-1.288029,0.724367,0.797056,0.027413,-0.814014,-0.836375,-2.328202,0.152464,1.606288
2010-02,1.473980,0.250654,-0.723416,-0.661578,0.917638,0.309648,-0.024901,-0.847392,-0.762782,0.515936,0.508775,0.653715,0.390610,0.295294
2010-03,1.473980,0.250951,-0.723416,0.239712,-0.225132,0.648425,1.529765,1.345209,1.654599,0.680700,-1.189698,1.388989,0.231846,0.095113
2010-04,0.395023,0.255791,-0.766140,-0.854091,1.188670,0.239296,-1.418214,0.031971,-1.266938,0.716863,-2.321022,0.809615,-0.006300,-0.752715
2010-05,0.395023,0.267080,-0.723416,0.197961,-1.076122,0.525682,1.555232,0.459296,0.588214,-0.229368,1.918011,-1.321816,-0.006300,0.291369
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-08,0.241495,-0.394589,0.691786,0.359348,-0.462913,3.742079,0.077822,-0.522541,-0.856335,-0.291604,1.565526,-1.170412,-0.720739,-3.932404
2019-09,0.241495,-0.566423,1.099995,0.249189,-0.463236,1.502679,-0.348135,0.175606,1.679065,-0.652859,0.033465,0.608406,-0.879503,-0.516340
2019-10,-1.745312,-1.027303,1.271875,-0.079969,0.195651,0.709595,0.408101,0.389583,-1.322661,0.409611,-1.119686,0.757678,-2.625908,-0.271126


,target,TWT.VADERstanceweight_log_stl_PE,TWT.CR_lognorm_PU+,ECN.PPI_MoMlog,ECN.CPI_MoMlog,ECN.IPI_MoMlog,ECN.exports_MoMlog,ECN.imports_MoMlog,ECN.govtexpt_MoMlog,ECN.pseiclose_MoMlog,ECN.phpusd_MoMlog,ECN.m1_MoMlog,ECN.tbill_usd90_diff,ECN.tdr_php360_diff
date,,,,,,,,,,,,,,
2019-07,0.241495,-0.184309,0.103097,-0.116413,0.200488,-4.426457,0.159613,1.040206,0.887537,0.002348,-1.231695,0.296748,-0.323828,-0.920180
2019-08,0.241495,-0.394589,0.691786,0.359348,-0.462913,3.742079,0.077822,-0.522541,-0.856335,-0.291604,1.565526,-1.170412,-0.720739,-3.932404
2019-09,0.241495,-0.566423,1.099995,0.249189,-0.463236,1.502679,-0.348135,0.175606,1.679065,-0.652859,0.033465,0.608406,-0.879503,-0.516340
2019-10,-1.745312,-1.027303,1.271875,-0.079969,0.195651,0.709595,0.408101,0.389583,-1.322661,0.409611,-1.119686,0.757678,-2.625908,-0.271126
2019-11,-1.745312,-2.339143,0.990975,0.065838,0.192771,-0.258566,-1.163958,-0.571077,0.686009,-0.754721,-1.450612,1.028405,0.390610,0.908627
2019-12,-1.745312,-3.676277,0.152549,0.176189,1.170612,-0.500816,0.217392,-0.536016,1.305292,0.087724,0.017363,1.917296,-0.323828,0.935148
2020-01,-4.572383,-4.502113,2.030664,0.102431,1.483688,0.369509,0.085905,0.812067,-2.699969,-1.837720,0.073619,-1.201478,-0.006300,-4.686276
2020-02,-4.572383,-4.698235,1.973626,0.213087,-2.093101,0.140767,-0.637016,-2.214641,-0.504857,-1.358139,-0.224887,-0.356517,-2.149616,-1.144424
2020-03,-4.572383,-4.343888,1.847731,-0.720139,-1.445246,-1.374051,-0.650282,0.470834,1.341500,-5.229900,0.231129,2.734071,-9.055854,0.559866


Train X shape:  torch.Size([38, 9, 13]) Train Y shape:  torch.Size([38, 3, 1]) Train Y out shape:  torch.Size([38, 3, 1])
Val X shape:  torch.Size([1, 9, 13]) Val Y shape:  torch.Size([1, 3, 1]) Val Y out shape:  torch.Size([1, 3, 1])


In [47]:
train_loader = DataLoader(train_dataset, batch_size=200, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=200, shuffle=False)
pl.seed_everything(42, workers=True)

lightning_model = STMFSeq2OneLightning(
    dim_x=train_loader.dataset.tensors[0].shape[-1],
    dim_y=train_loader.dataset.tensors[1].shape[-1],
    num_layers=2,
    learning_rate=0.001296,
    weight_decay= 0.000296285
)
checkpoint_callback = ModelCheckpoint(monitor="val_loss_y", mode="min", save_top_k=1, save_last=True)
lr_monitor = LearningRateMonitor(logging_interval='step')
# logger = TensorBoardLogger("lightning_logs")
# Find learning rate and train model with new learning rate
trainer = pl.Trainer(max_epochs=150, accelerator='cpu',log_every_n_steps=1, callbacks=[checkpoint_callback,lr_monitor],
                     deterministic=True, num_sanity_val_steps=0)
# tuner = Tuner(trainer)
# lr_finder = tuner.lr_find(lightning_model, train_loader)
# lightning_model.hparams.lr = lr_finder.suggestion()
trainer.fit(lightning_model, train_loader, val_loader)

Seed set to 42
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name      | Type        | Params | Mode 
--------------------------------------------------
0 | model     | STMFSeq2One | 2.4 K  | train
1 | criterion | MSELoss     | 0      | train
--------------------------------------------------
2.4 K     Trainable params
0         Non-trainable params
2.4 K     Total params
0.010     Total estimated model params size (MB)
25        Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=150` reached.


In [ ]:
# # trainer.checkpoint_callback.last_model_path
# trainer = pl.Trainer(
#     max_epochs=200,
#     accelerator='auto',
#     log_every_n_steps=1,
# )
# trainer.fit(lightning_model, train_loader, ckpt_path = 'lightning_logs/version_50/checkpoints/epoch=99-step=100.ckpt')

In [34]:
lightning_model = STMFSeq2OneLightning.load_from_checkpoint(
    checkpoint_path="lightning_logs/version_0/checkpoints/epoch=92-step=93.ckpt",
    model=STMFSeq2One(dim_x=valX_in.shape[-1], dim_y=valY_in.shape[-1], num_layers=2)
)
lightning_model.eval()
with torch.no_grad():
    y_pred = lightning_model(valX_in, valY_in)
    print(y_pred[0])
    print(target_scaler.inverse_transform(y_pred[0].cpu().numpy()))

tensor([[ 0.4504],
        [ 0.8271],
        [ 0.3391],
        [-1.0833],
        [ 0.6667],
        [ 0.7450],
        [-0.1113],
        [-1.1977],
        [ 0.9093],
        [ 0.5647],
        [ 0.2361],
        [-1.7521],
        [-4.5766]])
[[  7.570424  ]
 [  9.223799  ]
 [  7.082096  ]
 [  0.8390794 ]
 [  8.519657  ]
 [  8.863398  ]
 [  5.105307  ]
 [  0.33709237]
 [  9.58451   ]
 [  8.071941  ]
 [  6.62999   ]
 [ -2.095923  ]
 [-14.492382  ]]


In [ ]:
kmpair = {'PE': ['CRVADER_BVN','CR_BxP_0'],'PU+': ['CRVADER_BVN','CR_BxP_0']} # kmpair = {'PE':['CR_B0']}
for vintage in pd.date_range(start="2017-01-31", end="2017-02-01", freq="ME"):
    print(train_model(config={'data_window': 12}, vintage=vintage, with_econ=True, with_tweets=False, kmpair = kmpair, task='singletask'))

In [ ]:
import multiprocess as mp
import functools
from mixfreqlstm.train_vintage import train_one_vintage
import pandas as pd

kmpair = {'PE': ['CRVADER_BVN','CR_BxP_0'],'PU+': ['CRVADER_BVN','CR_BxP_0']} # kmpair = {'PE':['CR_B0']}
device = 'cpu'
vintage_ids = list(pd.date_range(start="2017-01-31", end="2017-04-01", freq="ME"))
with mp.get_context('spawn').Pool(processes=3) as pool:
    results = pool.map(functools.partial(train_one_vintage, with_econ=True, with_tweets=False, data_window = 15, kmpair = kmpair, device=device), vintage_ids)
print("\n".join(results))

In [35]:
test_data, _, _, _ = data_model.load_data(vintage=vintage, window=1000, kmpair=kmpair, with_econ=with_econ, with_tweets=with_tweets, target_release_lag=True,scaled=False, extend=True, DFM_order=(1,0,1,0), optimize_order = False)
test_data = test_data.loc[vintage + relativedelta(months = -((vintage.month - 1) % 3) - data_window - (3 if vintage.month % 3 == 1 else 0), day=31):,:]#.dropna()#.reset_index()  # get first month of same qtr last year, but get final day. Based on window = 12
test_data.iloc[:,:1] = target_scaler.transform(test_data.iloc[:,:1])
if with_econ and with_tweets:
    econ_n_feat = econ_scaler.n_features_in_
    tweets_n_feat = tweets_scaler.n_features_in_
    test_data.iloc[:,1:tweets_n_feat+1] = tweets_scaler.transform(test_data.iloc[:,1:tweets_n_feat+1])
    test_data.iloc[:,tweets_n_feat+1:] = econ_scaler.transform(test_data.iloc[:,tweets_n_feat+1:])
elif with_econ:
    econ_n_feat = econ_scaler.n_features_in_
    test_data.iloc[:,1:econ_n_feat+1] = econ_scaler.transform(test_data.iloc[:,1:econ_n_feat+1])
elif with_tweets:
    tweets_n_feat = tweets_scaler.n_features_in_
    test_data.iloc[:,1:tweets_n_feat+1] = tweets_scaler.transform(test_data.iloc[:,1:tweets_n_feat+1])

display(test_data)

,target,ECN.PPI_MoMlog,ECN.CPI_MoMlog,ECN.IPI_MoMlog,ECN.exports_MoMlog,ECN.imports_MoMlog,ECN.govtexpt_MoMlog,ECN.pseiclose_MoMlog,ECN.phpusd_MoMlog,ECN.m1_MoMlog,ECN.tbill_usd90_diff,ECN.tdr_php360_diff
date,,,,,,,,,,,,
2017-04,0.837011,-0.143193,0.428479,-1.042587,-0.694680,-1.095099,-1.271644,0.861576,-0.811407,0.783970,0.311228,-0.269924
2017-05,0.837011,0.176189,-1.098761,0.482204,0.669731,1.517449,1.551993,0.358942,-0.059715,-0.571643,1.343195,0.440524
2017-06,0.837011,0.149444,-0.793644,0.089442,-0.469286,-1.321920,0.114834,-0.102106,-0.074251,-0.092916,0.390610,-0.057967
2017-07,0.353892,0.149423,-0.488526,-0.166191,0.217356,-0.173597,-0.477720,0.344787,1.383378,-0.501751,0.311228,0.240343
2017-08,0.353892,0.229700,-0.184231,0.693486,0.237915,1.742433,-0.905687,-0.274628,0.371937,-0.178674,-0.482592,-0.191422
2017-09,0.353892,0.095892,1.330709,-0.127523,0.011161,-1.025902,0.682778,0.435940,0.187069,0.032180,0.390610,0.130439
2017-10,-1.092030,0.363266,0.112732,0.391951,-0.118796,0.822361,-0.230245,0.373974,0.543015,-0.141407,0.708138,0.259968
2017-11,-1.092030,-0.091367,0.711615,0.246662,-0.555165,0.373397,0.432933,-0.399352,-0.601845,1.224497,0.946284,0.134364
2017-12,-1.092030,0.122435,0.106312,0.916411,-0.350896,0.479319,1.164056,0.642124,-1.219753,1.181793,0.946284,1.535636


In [39]:
testX_in, testY_in, _, _ = sliding_windows_ST(test_data, train=False, seq_length=data_window)
model = STMFSeq2One(dim_x=testX_in.shape[-1], dim_y=testY_in.shape[-1], num_layers=2) # number of features in X and Y
lightning_model = STMFSeq2OneLightning.load_from_checkpoint(checkpoint_path='lightning_logs/version_0/checkpoints/epoch=92-step=93.ckpt', model=model).to('cpu')
lightning_model.eval()
with torch.no_grad():
    # start_row=0
    start_row = 3 if vintage.month % 3 == 1 else 0       # start_row = 3 if vintage.month % 3 == 1 else 0 ### whether to start from the first row or the fourth row
    if vintage.month % 3 == 1:
        print(testY_in[:,:-1])
        backcastY = lightning_model(testX_in[:,:-3],testY_in[:,:-1])
        testY_in[:,-2] = backcastY[0][-1]
        print(backcastY)
        # print(target_scaler.inverse_transform(backcastY[0].cpu().numpy()))
    print(testY_in[:,start_row//3:])
    nowcastY = lightning_model(testX_in[:,start_row:],testY_in[:,start_row//3:])
    print(nowcastY)
    nowcastY = target_scaler.inverse_transform(nowcastY[0].cpu().numpy())
    # display(pd.DataFrame({
    #     'Nowcast Y': nowcastY.flatten(),
    # }))

tensor([[[ 0.8370],
         [ 0.3539],
         [-1.0920],
         [ 0.6883],
         [ 0.7328],
         [-0.0845],
         [-1.2228],
         [ 0.9043],
         [ 0.6020],
         [ 0.2415],
         [-1.7453],
         [-4.5724],
         [    nan]]])
tensor([[[ 0.8897],
         [ 0.3286],
         [-1.1020],
         [ 0.6729],
         [ 0.7452],
         [-0.1112],
         [-1.1979],
         [ 0.9097],
         [ 0.5647],
         [ 0.2361],
         [-1.7521],
         [-3.9750],
         [-5.7157]]])


## Evaluation

In [19]:
# target = 'PHL_GDP_SA'
# vintage = pd.to_datetime('2020-06-30')
# kmpair = {'PE': ['CRVADER_BVN','CR_BxP_0'],'PU+': ['CRVADER_BVN','CR_BxP_0']} # kmpair = {'PE':['CR_B0']}
# data_window = 12
# with_econ = True
# with_tweets = False
evaluate_one_vintage_ST(vintage=vintage, with_econ=with_econ, with_tweets=with_tweets, kmpair = kmpair, target=target, config={'data_window': data_window, 'num_layers': 2},
                        ckpt_path='lightning_logs/version_2/checkpoints/epoch=67-step=68.ckpt', device='cpu')

(Timestamp('2020-06-30 00:00:00'), np.float32(7.986415))

In [25]:
# import multiprocess as mp
# import functools
# from mixfreqlstm.evaluate_vintage import evaluate_one_vintage
# from mixfreqlstm.data_utils import sliding_windows, NowcastingLSTM_MQ

kmpair = {'PE': ['CRVADER_BVN','CR_BxP_0'],'PU+': ['CRVADER_BVN','CR_BxP_0']} # kmpair = {'PE':['CR_B0']}
vintage_ids = list(pd.date_range(start="2017-12-31", end="2019-01-01", freq="ME"))
results = []
for vintage in vintage_ids:
    result = evaluate_one_vintage(vintage, with_econ=False, with_tweets=True, data_window = 15, kmpair = kmpair, version='-v5')
    results.append(result)
# with mp.get_context('spawn').Pool(processes=3) as pool: #.get_context('spawn')
#     results = pool.map(functools.partial(evaluate_one_vintage, device=device), vintage_ids)


results_df = pd.DataFrame(results, columns=["Vintage", "Prediction"])
results_df["Prediction"] = results_df["Prediction"].apply(lambda x: x[0][0])  # Extract the scalar value from the array
results_df = results_df.set_index("Vintage")
gdp = pd.read_csv('Results/Original/DFM_Opt_W1000_GDP_E_summary.csv', parse_dates=['date'], index_col=[0])[['Actual_Q']]
results_df = pd.concat([results_df, gdp], axis=1)
# results_df.to_csv('results_T_e150_w27.csv')
display(results_df.dropna())

/home/btiu/Documents/Research/TweetsNowcast/.conda/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/btiu/Documents/Research/TweetsNowcast/.conda/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,Prediction,Actual_Q
2017-12-31,7.184227,6.641206
2018-01-31,5.970207,6.454560
2018-02-28,5.487491,6.454560
2018-03-31,5.306432,6.454560
2018-04-30,6.606155,6.378895
2018-05-31,6.131291,6.378895
2018-06-30,6.117940,6.378895
2018-07-31,6.149463,6.145971
2018-08-31,5.351914,6.145971
2018-09-30,5.896016,6.145971


# Dual output LSTM

In [6]:
target = 'PHL_GDP_SA'
vintage = pd.to_datetime('2017-01-31')
# kmpair = {'PE': ['CR_G0','CR_B0']}
kmpair = {'PE': ['VADERstanceweight_log_stl'], 'PU+': ['CR_lognorm']}
with_econ = True
with_tweets = False
data_window = 72

## Train

In [7]:
### Train & Val Data
data_model = NowcastingLSTM_MQ()
data, target_scaler, econ_scaler, tweets_scaler = data_model.load_data(vintage=vintage,window=1000, kmpair=kmpair, with_econ=with_econ, with_tweets=with_tweets, target_release_lag=True, scaled=True, extend=False, DFM_order=(1,0,1,0), optimize_order = False, target=target)
data = data.dropna()
train_data = data.iloc[:-3]        # train_data = train_data.dropna(subset=train_data.columns.drop('target'))
val_data = data.iloc[-data_window-3:]

# val_data, _, _, _ = data_model.load_data(vintage=vintage+pd.offsets.QuarterEnd(0)+pd.DateOffset(months=2),window=1000, kmpair=kmpair, with_econ=with_econ, with_tweets=with_tweets, target_release_lag=False, scaled=False, extend=False, DFM_order=(1,0,1,0), optimize_order = False, target=target)
# val_data = val_data.dropna().iloc[-data_window-3:]
# # display(val_data)
# val_data.iloc[:,:1] = target_scaler.transform(val_data.iloc[:,:1])
# if with_econ and with_tweets:
#     econ_n_feat = econ_scaler.n_features_in_
#     tweets_n_feat = tweets_scaler.n_features_in_
#     val_data.iloc[:,1:tweets_n_feat+1] = tweets_scaler.transform(val_data.iloc[:,1:tweets_n_feat+1])
#     val_data.iloc[:,tweets_n_feat+1:] = econ_scaler.transform(val_data.iloc[:,tweets_n_feat+1:])
# elif with_econ:
#     econ_n_feat = econ_scaler.n_features_in_
#     val_data.iloc[:,1:econ_n_feat+1] = econ_scaler.transform(val_data.iloc[:,1:econ_n_feat+1])
# elif with_tweets:
#     tweets_n_feat = tweets_scaler.n_features_in_
#     val_data.iloc[:,1:tweets_n_feat+1] = tweets_scaler.transform(val_data.iloc[:,1:tweets_n_feat+1])
print('train_data DF:', len(train_data), 'val_data DF:',len(val_data))
display(train_data)
display(val_data)
trainX_in, trainY_in, trainX_out, trainY_out = sliding_windows_MT(train_data, train=True, seq_length=data_window)
valX_in, valY_in, valX_out, valY_out = sliding_windows_MT(val_data, train=True, seq_length=data_window)
# print("trainX_in length:", len(trainX_in), "valX_in length:", len(valX_in))
print("Train X shape: ", trainX_in.shape, "Train Y shape: ", trainY_in.shape, "Train X out shape:", trainX_out.shape,"Train Y out shape: ", trainY_out.shape)
print("Val X shape: ", valX_in.shape, "Val Y shape: ", valY_in.shape, "Val X out shape:", valX_out.shape, "Val Y out shape: ", valY_out.shape)
train_dataset = TensorDataset(trainX_in, trainY_in, trainX_out, trainY_out)
val_dataset = TensorDataset(valX_in, valY_in, valX_out, valY_out)
train_loader = DataLoader(train_dataset, batch_size=200, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=200, shuffle=False)

# train_loader, val_loader, _,_,_ = get_dataloader_for_vintage(vintage, with_econ, with_tweets, data_window = data_window, kmpair = kmpair, task='multitask')
# print(len(train_loader.dataset), len(val_loader.dataset))# Get the dimensions from train_loader
# dim_x = train_loader.dataset.tensors[0].shape[-1]
# dim_y = train_loader.dataset.tensors[1].shape[-1]

train_data DF: 78 val_data DF: 75


,target,ECN.PPI_MoMlog,ECN.CPI_MoMlog,ECN.IPI_MoMlog,ECN.exports_MoMlog,ECN.imports_MoMlog,ECN.govtexpt_MoMlog,ECN.pseiclose_MoMlog,ECN.phpusd_MoMlog,ECN.m1_MoMlog,ECN.tbill_usd90_diff,ECN.tdr_php360_diff
date,,,,,,,,,,,,
2010-01,2.129143,-0.658842,0.099338,-3.211650,0.670537,1.022208,0.022550,-0.980825,-0.801290,-2.245431,0.315864,2.257637
2010-02,2.129143,-1.842352,1.028094,0.521803,-0.069046,-1.302226,-0.810194,0.454662,0.458866,0.657919,0.965424,0.419876
2010-03,2.129143,0.441132,-0.217059,1.313458,1.465525,1.797025,1.737353,0.632501,-1.132289,1.373820,0.532384,0.139260
2010-04,0.401282,-2.330098,1.323408,0.357406,-1.444350,-0.059242,-1.341497,0.671534,-2.192131,0.809712,-0.117175,-1.049232
2010-05,0.401282,0.335353,-1.144291,1.026632,1.490663,0.544784,0.613548,-0.349785,1.779060,-1.265561,-0.117175,0.414374
...,...,...,...,...,...,...,...,...,...,...,...,...
2016-02,0.307705,0.545145,-1.523370,0.643788,0.191452,-2.411173,-0.360925,-0.286142,0.156055,0.089816,0.099344,0.205287
2016-03,0.307705,-0.717439,-0.836505,1.169044,0.729251,1.957813,1.318743,1.694746,-1.731107,1.779517,-2.715415,0.265812
2016-04,0.587631,0.680573,0.193314,-0.785956,-0.988310,0.245066,-0.944696,-0.553654,-0.881892,-0.122769,0.099344,-0.488000


,target,ECN.PPI_MoMlog,ECN.CPI_MoMlog,ECN.IPI_MoMlog,ECN.exports_MoMlog,ECN.imports_MoMlog,ECN.govtexpt_MoMlog,ECN.pseiclose_MoMlog,ECN.phpusd_MoMlog,ECN.m1_MoMlog,ECN.tbill_usd90_diff,ECN.tdr_php360_diff
date,,,,,,,,,,,,
2010-07,-1.204155,0.225500,0.390041,-0.040202,-0.153073,1.184179,0.273005,0.131697,-0.037081,-0.318345,-0.766735,-0.168867
2010-08,-1.204155,-0.433844,0.690969,0.418523,0.509917,-0.701755,-1.177230,0.673006,-2.206973,-0.022397,-0.333695,0.331840
2010-09,-1.204155,-0.217485,-0.836505,-0.335015,1.026167,0.276120,0.653123,2.931606,-1.738242,0.811157,0.315864,2.059555
2010-10,-0.665370,0.002509,-1.141494,1.195110,-1.087066,0.701870,-0.632224,0.683800,-1.771280,0.741569,-0.983255,-1.461903
2010-11,-0.665370,0.001649,2.504602,0.137886,-1.420057,0.034898,0.042143,-1.967447,0.022979,-0.375022,0.965424,0.166771
...,...,...,...,...,...,...,...,...,...,...,...,...
2016-05,0.587631,0.413256,-0.151551,0.147327,1.095954,0.265547,0.547489,0.524490,0.886301,-0.425965,2.481064,-0.031311
2016-06,0.587631,0.280191,0.529602,0.069233,0.043273,0.208867,0.027384,0.947426,-0.692230,-0.239036,-1.849335,-0.157863
2016-07,-0.299997,-0.722633,-0.155343,-0.031829,-0.214467,-0.236707,-0.041641,0.250063,1.021151,-0.467314,0.099344,-0.317429


Train X shape:  torch.Size([6, 75, 11]) Train Y shape:  torch.Size([6, 25, 1]) Train X out shape: torch.Size([6, 75, 11]) Train Y out shape:  torch.Size([6, 25, 1])
Val X shape:  torch.Size([3, 75, 11]) Val Y shape:  torch.Size([3, 25, 1]) Val X out shape: torch.Size([3, 75, 11]) Val Y out shape:  torch.Size([3, 25, 1])


In [ ]:
logger_enabled = True
device = 'cpu'
config = {
    'data_window': data_window,
    'num_layers': 1,
    'learning_rate': 1e-1,
    'weight_decay': 0,
    'alpha': 20,
    'epochs': 150}
pl.seed_everything(42, workers=True)
# train_loader, val_loader, _,_,_ = get_dataloader_for_vintage(vintage, with_econ, with_tweets, kmpair=kmpair, data_window=config['data_window'], task='multitask')
model = MTMFSeq2OneLightningOLD(
    dim_x=train_loader.dataset.tensors[0].shape[-1], # type: ignore
    dim_y=train_loader.dataset.tensors[1].shape[-1], # type: ignore
    num_layers=config['num_layers'],
    learning_rate=config["learning_rate"],
    weight_decay=config["weight_decay"],
    alpha=config['alpha']
)
checkpoint_callback = ModelCheckpoint(monitor="val_loss_y", mode="min", save_top_k=1, save_last=True)
lr_monitor = LearningRateMonitor(logging_interval='step')
trainer = pl.Trainer(
    max_epochs=config['epochs'],
    callbacks=[TuneReportCheckpointCallback(metrics={'val_loss_y': 'val_loss_y', 'val_loss_x': 'val_loss_x', 'train_loss': 'train_loss'})] + ([checkpoint_callback, lr_monitor] if logger_enabled else []),
    enable_checkpointing=True,
    deterministic=True,
    accelerator=device,
    num_sanity_val_steps=0,
    enable_progress_bar=logger_enabled,
    enable_model_summary=logger_enabled,
    logger=logger_enabled,
    log_every_n_steps=1 if logger_enabled else 50
)
trainer.fit(model, train_loader, val_loader)

In [10]:
lightning_model = MTMFSeq2OneLightningOLD.load_from_checkpoint(
    checkpoint_path="lightning_logs/version_6/checkpoints/epoch=20-step=21.ckpt",
    model=MTMFSeq2One(dim_x=valX_in.shape[-1], dim_y=valY_in.shape[-1])
)
lightning_model.eval()

with torch.no_grad():
    x_pred, y_pred = lightning_model(valX_in, valY_in)
    y_out = []
    for i in range(len(y_pred)):
        y_out.append(target_scaler.inverse_transform(y_pred[i].cpu().numpy()))
# print(x_pred[0][-3:], x_pred[1][-3:], x_pred[2][-3:])
print("y_pred:", y_out[0][-1], y_out[1][-1], y_out[2][-1])

y_pred: [6.921115] [6.953386] [7.0787477]


In [8]:
y_out[2]

array([[4.9177203],
       [4.2980256],
       [3.8765051],
       [3.55279  ],
       [3.4587812]], dtype=float32)

## Test

In [23]:
vintage= pd.to_datetime('2020-10-31')
data_window = 72

data_model = NowcastingLSTM_MQ()
_, target_scaler, econ_scaler, tweets_scaler = data_model.load_data(vintage=vintage,window=1000, kmpair=kmpair, with_econ=with_econ, with_tweets=with_tweets, target_release_lag=True,scaled=True, extend=False, DFM_order=(1,0,1,0), optimize_order = False, target=target)
test_data, _, _, _ = data_model.load_data(vintage=vintage, window=1000, kmpair=kmpair, with_econ=with_econ, with_tweets=with_tweets, target_release_lag=True,scaled=False, extend=False, DFM_order=(1,0,1,0), optimize_order = False, target=target)
test_data = test_data.loc[vintage + relativedelta(months = -((vintage.month - 1) % 3) - data_window - (3 if vintage.month % 3 == 1 else 0), day=31):,:]
test_data.iloc[:,:1] = target_scaler.transform(test_data.iloc[:,:1])

if with_econ and with_tweets:
    econ_n_feat = econ_scaler.n_features_in_
    tweets_n_feat = tweets_scaler.n_features_in_
    test_data.iloc[:,1:tweets_n_feat+1] = tweets_scaler.transform(test_data.iloc[:,1:tweets_n_feat+1])
    test_data.iloc[:,tweets_n_feat+1:] = econ_scaler.transform(test_data.iloc[:,tweets_n_feat+1:])
elif with_econ:
    econ_n_feat = econ_scaler.n_features_in_
    test_data.iloc[:,1:econ_n_feat+1] = econ_scaler.transform(test_data.iloc[:,1:econ_n_feat+1])
elif with_tweets:
    tweets_n_feat = tweets_scaler.n_features_in_
    test_data.iloc[:,1:tweets_n_feat+1] = tweets_scaler.transform(test_data.iloc[:,1:tweets_n_feat+1])
display(test_data)
print(len(test_data))

,target,TWT.VADERstanceweight_log_stl_PE,TWT.CR_lognorm_PU+,ECN.PPI_MoMlog,ECN.CPI_MoMlog,ECN.IPI_MoMlog,ECN.exports_MoMlog,ECN.imports_MoMlog,ECN.govtexpt_MoMlog,ECN.pseiclose_MoMlog,ECN.phpusd_MoMlog,ECN.m1_MoMlog,ECN.tbill_usd90_diff,ECN.tdr_php360_diff
date,,,,,,,,,,,,,,
2014-07,0.025845,0.277022,-0.687307,0.055623,1.112601,-0.170612,-0.098859,1.108186,-0.826786,-0.054930,-0.770272,0.117455,-0.082378,0.006176
2014-08,0.025845,0.248649,-0.399579,0.607756,-0.163712,0.037271,0.041701,0.356056,-0.859794,0.444879,0.603411,-0.625714,-0.001858,0.187593
2014-09,0.025845,0.197793,-0.441526,0.461009,-0.164896,0.249363,0.583181,-0.128565,0.551768,0.563869,0.613672,-1.468891,-0.082378,0.428167
2014-10,0.130599,0.166208,0.213482,-0.228290,-1.114847,0.627771,-1.226329,-0.318350,-0.176434,-0.313365,1.464982,-0.054339,-0.082378,0.025895
2014-11,0.130599,0.172311,-0.564028,-0.257080,-1.432286,-0.233940,0.011509,-0.119767,-0.134284,0.110229,0.283806,0.521244,0.078662,0.763394
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-06,-5.872272,-1.448027,1.309619,-0.173864,1.158492,-3.228497,1.808623,1.355069,-0.091304,1.171230,-0.868279,-0.624775,0.159182,-0.499142
2020-07,NaN,-0.443406,2.482103,0.097834,0.823790,-1.284446,0.241532,0.921842,0.279664,-1.086520,-1.193845,0.668625,-0.565499,-1.518517
2020-08,NaN,0.106937,1.425418,0.215192,-1.122093,0.068791,-0.369639,-0.203648,-1.285104,-0.275620,-1.198392,-0.038147,0.159182,-0.367257


76


In [69]:
testX_in, testY_in, _, _ = sliding_windows_MT(test_data, train = False, seq_length=data_window)

model = MTMFSeq2One(dim_x=testX_in.shape[-1], dim_y=testY_in.shape[-1])
lightning_model = MTMFSeq2OneLightning.load_from_checkpoint(checkpoint_path=f"lightning_logs/version_0/checkpoints/epoch=99-step=100.ckpt", model=model)
lightning_model.eval()
with torch.no_grad():
    # start_row = 3 if vintage.month % 3 == 1 else 0
    start_row = 0
    if vintage.month % 3 == 1:
        latent = lightning_model.model.encoder_x(testX_in[:,start_row:-3]) # Encode the input sequence excluding the last 3 columns
        backcastX = lightning_model.model.decoder_x(latent)
        nan_mask = torch.isnan(testX_in[0, -4])                 # Find NaNs in the second to the last row of testX_in
        testX_in[0, -4][nan_mask] = backcastX[0, -1][nan_mask]  # Take last row of backcast and replace NaNs
        _, backcastY = lightning_model(testX_in[:,start_row:-3],testY_in[:,:-1])
        testY_in[:,-2] = backcastY[0,-1]
        print("Backcast Y:", backcastY[0,-1].cpu().numpy())
    # display(pd.DataFrame(testX_in[0][:,start_row:], columns=test_data.columns[1:]))
    # display(pd.DataFrame(testY_in[0][:,start_row//3:], columns=test_data.columns[:1]))
    nowcastX_0, nowcastY_0 = lightning_model(testX_in[:,start_row:],testY_in[:,start_row//3:])
    nan_mask = torch.isnan(testX_in[0, -3:])                    # Find NaNs in the current quarter of testX_in
    testX_in[0, -3:][nan_mask] = nowcastX_0[0,-3:][nan_mask]        # Take last row of backcast and replace NaNs
    nowcastX_1, nowcastY_1 = lightning_model(testX_in[:,start_row:],testY_in[:,start_row//3:])
    nowcastY_0 = target_scaler.inverse_transform(nowcastY_0[0].cpu().numpy())
    nowcastY_1 = target_scaler.inverse_transform(nowcastY_1[0].cpu().numpy())
    # Convert nowcastX_0 from 1, 15, 13] tensor to dataframe
    # nowcastX_0 = nowcastX_0[0].cpu().numpy()
    # nowcastX_0 = pd.DataFrame(nowcastX_0, columns=test_data.columns[1:])
    # display(nowcastX_0)
    display(pd.DataFrame({
        'Nowcast Y_0': nowcastY_0.flatten(),
        'Nowcast Y_1': nowcastY_1.flatten()
    }))
    # print("Nowcast Y_1:", nowcastY_1.flatten()[-1])
    
    

Backcast Y: [-2.4978182]


,Nowcast Y_0,Nowcast Y_1
0,5.554373,5.554373
1,5.204476,5.204476
2,7.168970,7.168970
3,7.821695,7.821695
4,3.964226,3.964226
5,8.252216,8.252216
6,6.592866,6.592866
7,8.316326,8.316326
8,5.495874,5.495874
9,5.627817,5.627817


## Dual Output NO GDP

In [16]:
target = 'PHL_GDP_SA'
vintage = pd.to_datetime('2017-01-31')
kmpair = {'PE': ['CR_G0','CR_B0']}
with_econ = True
with_tweets = False
data_window = 12

In [17]:
### Train & Val Data
data_model = NowcastingLSTM_MQ()
data, target_scaler, econ_scaler, tweets_scaler = data_model.load_data(vintage=vintage,window=1000, kmpair=kmpair, with_econ=with_econ, with_tweets=with_tweets, target_release_lag=True, scaled=True, extend=False, DFM_order=(1,0,1,0), optimize_order = False, target=target)
data = data.dropna()
train_data = data.iloc[:-3]        # train_data = train_data.dropna(subset=train_data.columns.drop('target'))
val_data = data.iloc[-data_window-3:]

# val_data, _, _, _ = data_model.load_data(vintage=vintage+pd.offsets.QuarterEnd(0)+pd.DateOffset(months=2),window=1000, kmpair=kmpair, with_econ=with_econ, with_tweets=with_tweets, target_release_lag=False, scaled=False, extend=False, DFM_order=(1,0,1,0), optimize_order = False, target=target)
# val_data = val_data.dropna().iloc[-data_window-3:]
# # display(val_data)
# val_data.iloc[:,:1] = target_scaler.transform(val_data.iloc[:,:1])
# if with_econ and with_tweets:
#     econ_n_feat = econ_scaler.n_features_in_
#     tweets_n_feat = tweets_scaler.n_features_in_
#     val_data.iloc[:,1:tweets_n_feat+1] = tweets_scaler.transform(val_data.iloc[:,1:tweets_n_feat+1])
#     val_data.iloc[:,tweets_n_feat+1:] = econ_scaler.transform(val_data.iloc[:,tweets_n_feat+1:])
# elif with_econ:
#     econ_n_feat = econ_scaler.n_features_in_
#     val_data.iloc[:,1:econ_n_feat+1] = econ_scaler.transform(val_data.iloc[:,1:econ_n_feat+1])
# elif with_tweets:
#     tweets_n_feat = tweets_scaler.n_features_in_
#     val_data.iloc[:,1:tweets_n_feat+1] = tweets_scaler.transform(val_data.iloc[:,1:tweets_n_feat+1])
print('train_data DF:', len(train_data), 'val_data DF:',len(val_data))
display(train_data)
display(val_data)
trainX_in, _, trainX_out, trainY_out = sliding_windows_MT(train_data, train=True, seq_length=data_window)
valX_in, _, valX_out, valY_out = sliding_windows_MT(val_data, train=True, seq_length=data_window)
# print("trainX_in length:", len(trainX_in), "valX_in length:", len(valX_in))
print("Train X shape: ", trainX_in.shape, "Train X out shape:", trainX_out.shape,"Train Y out shape: ", trainY_out.shape)
print("Val X shape: ", valX_in.shape, "Val X out shape:", valX_out.shape, "Val Y out shape: ", valY_out.shape)
train_dataset = TensorDataset(trainX_in, trainX_out, trainY_out)
val_dataset = TensorDataset(valX_in, valX_out, valY_out)
train_loader = DataLoader(train_dataset, batch_size=200, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=200, shuffle=False)

# train_loader, val_loader, _,_,_ = get_dataloader_for_vintage(vintage, with_econ, with_tweets, data_window = data_window, kmpair = kmpair, task='multitask')
# print(len(train_loader.dataset), len(val_loader.dataset))# Get the dimensions from train_loader
# dim_x = train_loader.dataset.tensors[0].shape[-1]
# dim_y = train_loader.dataset.tensors[1].shape[-1]

train_data DF: 78 val_data DF: 15


,target,ECN.PPI_MoMlog,ECN.CPI_MoMlog,ECN.IPI_MoMlog,ECN.exports_MoMlog,ECN.imports_MoMlog,ECN.govtexpt_MoMlog,ECN.pseiclose_MoMlog,ECN.phpusd_MoMlog,ECN.m1_MoMlog,ECN.tbill_usd90_diff,ECN.tdr_php360_diff
date,,,,,,,,,,,,
2010-01,2.129143,-0.658842,0.099338,-3.211650,0.670537,1.022208,0.022550,-0.980825,-0.801290,-2.245431,0.315864,2.257637
2010-02,2.129143,-1.842352,1.028094,0.521803,-0.069046,-1.302226,-0.810194,0.454662,0.458866,0.657919,0.965424,0.419876
2010-03,2.129143,0.441132,-0.217059,1.313458,1.465525,1.797025,1.737353,0.632501,-1.132289,1.373820,0.532384,0.139260
2010-04,0.401282,-2.330098,1.323408,0.357406,-1.444350,-0.059242,-1.341497,0.671534,-2.192131,0.809712,-0.117175,-1.049232
2010-05,0.401282,0.335353,-1.144291,1.026632,1.490663,0.544784,0.613548,-0.349785,1.779060,-1.265561,-0.117175,0.414374
...,...,...,...,...,...,...,...,...,...,...,...,...
2016-02,0.307705,0.545145,-1.523370,0.643788,0.191452,-2.411173,-0.360925,-0.286142,0.156055,0.089816,0.099344,0.205287
2016-03,0.307705,-0.717439,-0.836505,1.169044,0.729251,1.957813,1.318743,1.694746,-1.731107,1.779517,-2.715415,0.265812
2016-04,0.587631,0.680573,0.193314,-0.785956,-0.988310,0.245066,-0.944696,-0.553654,-0.881892,-0.122769,0.099344,-0.488000


,target,ECN.PPI_MoMlog,ECN.CPI_MoMlog,ECN.IPI_MoMlog,ECN.exports_MoMlog,ECN.imports_MoMlog,ECN.govtexpt_MoMlog,ECN.pseiclose_MoMlog,ECN.phpusd_MoMlog,ECN.m1_MoMlog,ECN.tbill_usd90_diff,ECN.tdr_php360_diff
date,,,,,,,,,,,,
2015-07,-0.606369,-0.491363,-0.146430,-0.556193,-0.032755,1.648963,-0.581533,-0.273368,0.466925,-0.636709,1.398464,-0.223890
2015-08,-0.606369,0.280191,-0.836505,-0.277898,-0.485875,-1.357857,-1.286785,-1.626437,1.579492,-1.037719,-0.117175,-0.009301
2015-09,-0.606369,0.280191,-1.526580,0.816969,-0.363342,0.063233,0.620298,-0.893448,1.055936,0.133847,-2.065855,0.381360
2015-10,0.889742,0.215595,-0.491306,0.777288,-0.783331,0.443344,0.049709,0.546578,-0.788701,-0.103913,1.831504,-0.064324
2015-11,0.889742,0.538298,0.197158,-0.291215,0.994455,-0.948426,-0.527672,-0.897823,1.119055,0.423199,2.914104,0.854556
2015-12,0.889742,0.086681,-0.149000,-0.779186,-0.940319,0.206451,1.506092,-0.148204,0.338062,1.674015,-1.416295,0.915081
2016-01,0.307705,-2.141334,-0.493232,-0.079503,-0.910612,0.998523,-1.220303,-1.108851,0.439658,-1.501138,3.347143,-0.994209
2016-02,0.307705,0.545145,-1.523370,0.643788,0.191452,-2.411173,-0.360925,-0.286142,0.156055,0.089816,0.099344,0.205287
2016-03,0.307705,-0.717439,-0.836505,1.169044,0.729251,1.957813,1.318743,1.694746,-1.731107,1.779517,-2.715415,0.265812


Train X shape:  torch.Size([66, 15, 11]) Train X out shape: torch.Size([66, 15, 11]) Train Y out shape:  torch.Size([66, 5, 1])
Val X shape:  torch.Size([3, 15, 11]) Val X out shape: torch.Size([3, 15, 11]) Val Y out shape:  torch.Size([3, 5, 1])


In [15]:
trainY_out

tensor([[[-1.2228],
         [ 0.9043],
         [ 0.6020],
         [ 0.2415],
         [-1.7453]],

        [[-1.2228],
         [ 0.9043],
         [ 0.6020],
         [ 0.2415],
         [-1.7453]],

        [[-1.2228],
         [ 0.9043],
         [ 0.6020],
         [ 0.2415],
         [-1.7453]]])

In [18]:
logger_enabled = True
device = 'cpu'
config = {
    'data_window': data_window,
    'num_layers': 2,
    'learning_rate': 1e-1,
    'weight_decay': 1e-3,
    'alpha': 20,
    'epochs': 150}
pl.seed_everything(42, workers=True)
# train_loader, val_loader, _,_,_ = get_dataloader_for_vintage(vintage, with_econ, with_tweets, kmpair=kmpair, data_window=config['data_window'], task='multitask')
model = MTSeq2OneMLightning(
    dim_x=train_loader.dataset.tensors[0].shape[-1], # type: ignore
    dim_y=train_loader.dataset.tensors[2].shape[-1], # type: ignore
    num_layers=config['num_layers'],
    learning_rate=config["learning_rate"],
    weight_decay=config["weight_decay"],
    alpha=config['alpha']
)
checkpoint_callback = ModelCheckpoint(monitor="val_loss_y", mode="min", save_top_k=1, save_last=True)
lr_monitor = LearningRateMonitor(logging_interval='step')
trainer = pl.Trainer(
    max_epochs=config['epochs'],
    callbacks=[TuneReportCheckpointCallback(metrics={'val_loss_y': 'val_loss_y', 'val_loss_x': 'val_loss_x', 'train_loss': 'train_loss'})] + ([checkpoint_callback, lr_monitor] if logger_enabled else []),
    enable_checkpointing=True,
    deterministic=True,
    accelerator=device,
    num_sanity_val_steps=0,
    enable_progress_bar=logger_enabled,
    enable_model_summary=logger_enabled,
    logger=logger_enabled,
    log_every_n_steps=1 if logger_enabled else 50
)
trainer.fit(model, train_loader, val_loader)

Seed set to 42
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name      | Type       | Params | Mode 
-------------------------------------------------
0 | model     | MTSeq2OneM | 2.0 K  | train
1 | criterion | MSELoss    | 0      | train
-------------------------------------------------
2.0 K     Trainable params
0         Non-trainable params
2.0 K     Total params
0.008     Total estimated model params size (MB)
27        Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=150` reached.


In [ ]:
lightning_model = MTSeq2OneMLightning.load_from_checkpoint(
    checkpoint_path="lightning_logs/version_1/checkpoints/epoch=0-step=1.ckpt",
    model=MTSeq2OneM(dim_x=valX_in.shape[-1], dim_y=valY_out.shape[-1])
)
lightning_model.eval()

with torch.no_grad():
    x_pred, y_pred = lightning_model(valX_in)
    y_out = []
    for i in range(len(y_pred)):
        y_out.append(target_scaler.inverse_transform(y_pred[i].cpu().numpy()))
# print(x_pred[0][-3:], x_pred[1][-3:], x_pred[2][-3:])
print("y_pred:", y_out[0][-1], y_out[1][-1], y_out[2][-1])

y_pred: [6.1165276] [6.1165276] [6.1165276]


In [8]:
y_out[2]

array([[6.1165276],
       [6.1165276],
       [6.1165276],
       [6.1165276],
       [6.1165276]], dtype=float32)